In [1]:
# 06-1. 기본 설정

from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

PROJECT_ROOT = Path(
    r"C:\code\portfolio_optimization"
)

SUPERVISED_DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "features"
    / "common"
    / "supervised_dataset.parquet"
)

print(
    SUPERVISED_DATASET_PATH.exists()
)

True


In [2]:
# 06-2. supervised dataset 불러오기

supervised_dataset = (
    pq.read_table(
        SUPERVISED_DATASET_PATH
    )
    .to_pandas()
    .sort_values(
        [
            "signal_date",
            "ticker"
        ]
    )
    .reset_index(
        drop=True
    )
)

print(
    "shape:",
    supervised_dataset.shape
)

print(
    "signals:",
    supervised_dataset[
        "signal_date"
    ].nunique()
)

print(
    "start:",
    supervised_dataset[
        "signal_date"
    ].min()
)

print(
    "end:",
    supervised_dataset[
        "signal_date"
    ].max()
)

shape: (21779, 39)
signals: 436
start: 2018-05-04 00:00:00
end: 2026-09-04 00:00:00


In [3]:
# 06-3. model feature 설정

ASSET_FEATURES = [
    "return_1d",
    "return_5d",
    "momentum_20d",
    "momentum_60d",
    "volatility_20d",
    "drawdown_20d",
    "trading_value_ma20",
    "trading_value_ratio_20d",
    "log_market_cap"
]

MARKET_FEATURES = [
    "market_return_1d",
    "market_return_5d",
    "market_return_20d",
    "market_volatility_20d",
    "market_drawdown",
    "volume_change_1d",
    "trading_value_change_1d",
    "market_trading_value_ratio_20d"
]

MACRO_FEATURES = [
    "base_rate",
    "usdkrw",
    "bond3y",
    "usdkrw_return_1d",
    "usdkrw_return_5d",
    "usdkrw_return_20d",
    "bond3y_change_1d",
    "bond3y_change_5d",
    "bond3y_change_20d",
    "base_rate_change",
    "rate_spread_3y"
]

MODEL_FEATURES = (
    ASSET_FEATURES
    + MARKET_FEATURES
    + MACRO_FEATURES
)

TARGET = "target_return"

print(
    "feature count:",
    len(MODEL_FEATURES)
)

feature count: 28


In [4]:
# 06-4. model dataset 품질 확인

print(
    "feature nan:",
    supervised_dataset[
        MODEL_FEATURES
    ]
    .isna()
    .sum()
    .sum()
)

print(
    "feature inf:",
    np.isinf(
        supervised_dataset[
            MODEL_FEATURES
        ]
    )
    .sum()
    .sum()
)

print(
    "target nan:",
    supervised_dataset[
        TARGET
    ]
    .isna()
    .sum()
)

print(
    "target inf:",
    np.isinf(
        supervised_dataset[
            TARGET
        ]
    )
    .sum()
)

feature nan: 0
feature inf: 0
target nan: 0
target inf: 0


In [5]:
# 06-5. target 분포 확인

print(
    supervised_dataset[
        TARGET
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
)

count    21779.000000
mean         0.001968
std          0.080628
min         -0.585680
1%          -0.186493
5%          -0.111018
25%         -0.039127
50%         -0.002803
75%          0.036001
95%          0.128183
99%          0.257860
max          1.154353
Name: target_return, dtype: float64


In [6]:
# 06-6. target 극단값 확인

target_extreme = (
    pd.concat(
        [
            supervised_dataset.nsmallest(
                10,
                TARGET
            ),
            supervised_dataset.nlargest(
                10,
                TARGET
            )
        ]
    )
    [
        [
            "signal_date",
            "execution_date",
            "ticker",
            "name",
            TARGET
        ]
    ]
)

print(
    target_extreme.to_string(
        index=False
    )
)

signal_date execution_date ticker    name  target_return
 2019-06-21     2019-06-24 028300   에이치엘비      -0.585680
 2019-09-20     2019-09-23 084990   헬릭스미스      -0.584912
 2019-07-26     2019-07-29 215600     신라젠      -0.530128
 2024-05-10     2024-05-13 028300     HLB      -0.528152
 2018-07-13     2018-07-16 007390    네이처셀      -0.519111
 2019-09-20     2019-09-23 073070     에스모      -0.511845
 2026-07-24     2026-07-27 475150  SK이터닉스      -0.428320
 2020-12-18     2020-12-21 069620    대웅제약      -0.411660
 2026-05-29     2026-06-01 011070   LG이노텍      -0.384749
 2023-05-04     2023-05-08 064550   바이오니아      -0.384661
 2020-03-20     2020-03-23 005690     파미셀       1.154353
 2020-07-17     2020-07-20 285130   SK케미칼       0.967039
 2018-08-31     2018-09-03 007390    네이처셀       0.796579
 2020-12-11     2020-12-14 069620    대웅제약       0.779196
 2020-03-20     2020-03-23 096530      씨젠       0.743852
 2026-04-17     2026-04-20 036930 주성엔지니어링       0.738505
 2026-05-22     2026-05-26 0110

과거 3년 train
        ->
다음 6개월 validation
        ->
다음 6개월 test
        ->
6개월 전진
        ->
train 기간 확대
        ->
다시 validation 6개월
        ->
test 6개월

signal
→ 다음 execution까지의 미래 수익률

In [7]:
# 06-7. walk-forward fold 생성

signal_start = (
    supervised_dataset[
        "signal_date"
    ].min()
)

signal_end = (
    supervised_dataset[
        "signal_date"
    ].max()
)


train_years = 3
validation_months = 6
test_months = 6
step_months = 6


folds = []

validation_start = (
    signal_start
    + pd.DateOffset(
        years=train_years
    )
)


fold_id = 1


while True:

    test_start = (
        validation_start
        + pd.DateOffset(
            months=validation_months
        )
    )

    test_end = (
        test_start
        + pd.DateOffset(
            months=test_months
        )
    )

    if test_end > signal_end:
        break

    folds.append(
        {
            "fold": fold_id,
            "train_start": signal_start,
            "train_end": validation_start,
            "validation_start": validation_start,
            "validation_end": test_start,
            "test_start": test_start,
            "test_end": test_end
        }
    )

    validation_start = (
        validation_start
        + pd.DateOffset(
            months=step_months
        )
    )

    fold_id += 1


fold_table = pd.DataFrame(
    folds
)

print(
    fold_table
)

   fold train_start  train_end validation_start validation_end test_start  \
0     1  2018-05-04 2021-05-04       2021-05-04     2021-11-04 2021-11-04   
1     2  2018-05-04 2021-11-04       2021-11-04     2022-05-04 2022-05-04   
2     3  2018-05-04 2022-05-04       2022-05-04     2022-11-04 2022-11-04   
3     4  2018-05-04 2022-11-04       2022-11-04     2023-05-04 2023-05-04   
4     5  2018-05-04 2023-05-04       2023-05-04     2023-11-04 2023-11-04   
5     6  2018-05-04 2023-11-04       2023-11-04     2024-05-04 2024-05-04   
6     7  2018-05-04 2024-05-04       2024-05-04     2024-11-04 2024-11-04   
7     8  2018-05-04 2024-11-04       2024-11-04     2025-05-04 2025-05-04   
8     9  2018-05-04 2025-05-04       2025-05-04     2025-11-04 2025-11-04   

    test_end  
0 2022-05-04  
1 2022-11-04  
2 2023-05-04  
3 2023-11-04  
4 2024-05-04  
5 2024-11-04  
6 2025-05-04  
7 2025-11-04  
8 2026-05-04  


In [8]:
# 06-8. fold별 sample 수 확인
# purge: 제거/ 아직 정답이 완전히 관측되지 않은 경계 데이터 제거

fold_summary = []


for fold in folds:

    train_mask = (
        (
            supervised_dataset["signal_date"]
            < fold["train_end"]
        )
        &
        (
            supervised_dataset["next_execution_date"]
            <= fold["train_end"]
        )
    )

    validation_mask = (
        (
            supervised_dataset["signal_date"]
            >= fold["validation_start"]
        )
        &
        (
            supervised_dataset["signal_date"]
            < fold["validation_end"]
        )
        &
        (
            supervised_dataset["next_execution_date"]
            <= fold["validation_end"]
        )
    )

    test_mask = (
        (
            supervised_dataset["signal_date"]
            >= fold["test_start"]
        )
        &
        (
            supervised_dataset["signal_date"]
            < fold["test_end"]
        )
    )


    fold_summary.append(
        {
            "fold": fold["fold"],
            "train_rows": train_mask.sum(),
            "validation_rows": validation_mask.sum(),
            "test_rows": test_mask.sum(),
            "train_signals": supervised_dataset.loc[
                train_mask,
                "signal_date"
            ].nunique(),
            "validation_signals": supervised_dataset.loc[
                validation_mask,
                "signal_date"
            ].nunique(),
            "test_signals": supervised_dataset.loc[
                test_mask,
                "signal_date"
            ].nunique()
        }
    )


fold_summary = pd.DataFrame(
    fold_summary
)

print(
    fold_summary.to_string(
        index=False
    )
)

 fold  train_rows  validation_rows  test_rows  train_signals  validation_signals  test_signals
    1        7791             1248       1300            156                  25            26
    2        9089             1250       1300            182                  25            26
    3       10389             1250       1298            208                  25            26
    4       11689             1248       1350            234                  25            27
    5       12987             1250       1297            260                  25            26
    6       14287             1197       1299            286                  24            26
    7       15584             1249       1298            312                  25            26
    8       16933             1198       1299            339                  24            26
    9       18181             1249       1299            364                  25            26


Ridge
→ StandardScaler
→ alpha tuning
→ validation
→ OOS test prediction
→ MSE / MAE / R² / IC

In [10]:
# 06-9. ridge model 패키지

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler #feature 단위 맞추기
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

In [11]:
# 06-10. fold 1 데이터 분리

fold = folds[0]

train_mask = (
    (supervised_dataset["signal_date"] < fold["train_end"])
    &
    (
        supervised_dataset["next_execution_date"]
        <= fold["train_end"]
    )
)

validation_mask = (
    (
        supervised_dataset["signal_date"]
        >= fold["validation_start"]
    )
    &
    (
        supervised_dataset["signal_date"]
        < fold["validation_end"]
    )
    &
    (
        supervised_dataset["next_execution_date"]
        <= fold["validation_end"]
    )
)

test_mask = (
    (
        supervised_dataset["signal_date"]
        >= fold["test_start"]
    )
    &
    (
        supervised_dataset["signal_date"]
        < fold["test_end"]
    )
)


train_df = supervised_dataset.loc[
    train_mask
].copy()

validation_df = supervised_dataset.loc[
    validation_mask
].copy()

test_df = supervised_dataset.loc[
    test_mask
].copy()


print(
    "train:",
    train_df.shape
)

print(
    "validation:",
    validation_df.shape
)

print(
    "test:",
    test_df.shape
)

train: (7791, 39)
validation: (1248, 39)
test: (1300, 39)


In [12]:
# 06-11. train validation 데이터 설정
# x = 입력 feature
# y = 모델이 맞혀야 할 정답

X_train = train_df[
    MODEL_FEATURES
]

y_train = train_df[
    TARGET
]

X_validation = validation_df[
    MODEL_FEATURES
]

y_validation = validation_df[
    TARGET
]

ridge의 핵심 parameter: alpha
alpha가 클수록: 계수를 더 강하게 억제 -> 모델 단순 -> 과적합 감소
작을수록: 일반 선형회귀에 가까워짐

In [13]:
# 06-12. ridge alpha 후보 설정

RIDGE_ALPHAS = [
    0.001,
    0.01,
    0.1,
    1.0,
    10.0,
    100.0,
    1000.0
]

IC: information coefficient: 모델이 매주 예측한 종목 순위와 실제 수익률 순위가 얼마나 비슷했는가?

보통 quant에서 spearman rank correlation을 많이 씀

In [16]:
# 06-13. cross-sectional ic 계산 함수
# 여기서 cross-sectional: 한 날짜에서 50개 종목을 서로 비교

def calculate_ic(
    data,
    prediction_column,
    target_column
):

    ic_values = []

    for _, group in data.groupby(
        "signal_date"
    ):

        if len(group) < 2:
            continue

        ic = (
            group[
                prediction_column
            ]
            .rank()
            .corr(
                group[
                    target_column
                ]
                .rank()
            )
        )

        if pd.notna(ic):
            ic_values.append(ic)

    return np.array(
        ic_values
    )

In [17]:
# 06-14. ridge alpha validation

ridge_validation_results = []


for alpha in RIDGE_ALPHAS:

    model = Pipeline(
        [
            (
                "scaler",
                StandardScaler()
            ),
            (
                "ridge",
                Ridge(
                    alpha=alpha
                )
            )
        ]
    )

    model.fit(
        X_train,
        y_train
    )

    validation_pred = model.predict(
        X_validation
    )

    validation_result = (
        validation_df[
            [
                "signal_date",
                "ticker",
                TARGET
            ]
        ]
        .copy()
    )

    validation_result[
        "prediction"
    ] = validation_pred


    mse = mean_squared_error(
        y_validation,
        validation_pred
    )

    rmse = np.sqrt(
        mse
    )

    mae = mean_absolute_error(
        y_validation,
        validation_pred
    )

    r2 = r2_score(
        y_validation,
        validation_pred
    )


    ic_values = calculate_ic(
        validation_result,
        "prediction",
        TARGET
    )


    ridge_validation_results.append(
        {
            "alpha": alpha,
            "rmse": rmse,
            "mae": mae,
            "r2": r2,
            "mean_ic": ic_values.mean(),
            "median_ic": np.median(
                ic_values
            ),
            "ic_signals": len(
                ic_values
            )
        }
    )


ridge_validation_results = pd.DataFrame(
    ridge_validation_results
)


print(
    ridge_validation_results.to_string(
        index=False
    )
)

   alpha     rmse      mae        r2   mean_ic  median_ic  ic_signals
   0.001 0.067651 0.047947 -0.066008 -0.004534  -0.059304          25
   0.010 0.067651 0.047947 -0.066007 -0.004534  -0.059304          25
   0.100 0.067651 0.047947 -0.066000 -0.004534  -0.059304          25
   1.000 0.067649 0.047943 -0.065927 -0.004424  -0.058343          25
  10.000 0.067626 0.047911 -0.065207 -0.005830  -0.051525          25
 100.000 0.067432 0.047626 -0.059092 -0.006048  -0.052485          25
1000.000 0.066652 0.046436 -0.034732  0.003301  -0.034526          25


RMSE
Root Mean Squared Error
= 평균제곱근오차
= 예측이 실제 수익률에서 얼마나 벗어났나
→ 낮을수록 좋음

MAE
Mean Absolute Error
= 평균절대오차
→ 낮을수록 좋음

R²
R-squared
= 결정계수
= 수익률 변동을 모델이 얼마나 설명했나
→ 높을수록 좋음

IC
Information Coefficient
= 예측 종목 순위와 실제 종목 순위의 상관
→ 양수일수록 예측 순위가 실제와 같은 방향

In [18]:
# 06-15. validation 기준 best alpha 선택

best_row = (
    ridge_validation_results
    .sort_values(
        "rmse"
    )
    .iloc[0]
)


best_alpha = (
    best_row[
        "alpha"
    ]
)


print(
    "best alpha:",
    best_alpha
)

print(
    best_row
)

best alpha: 1000.0
alpha         1000.000000
rmse             0.066652
mae              0.046436
r2              -0.034732
mean_ic          0.003301
median_ic       -0.034526
ic_signals      25.000000
Name: 6, dtype: float64


alpha를 크게 해서 모델을 단순하게 만들수록 성능이 나아진다 → 현재 feature의 선형 신호가 약하거나 noise가 많을 가능성

In [19]:
# 06-16. ridge alpha 범위 확장

RIDGE_ALPHAS = [
    0.001,
    0.01,
    0.1,
    1.0,
    10.0,
    100.0,
    1000.0,
    10000.0,
    100000.0,
    1000000.0
]

In [20]:
# 06-17. ridge validation 함수

def evaluate_ridge_alphas(
    train_df,
    validation_df,
    alphas
):

    X_train = train_df[
        MODEL_FEATURES
    ]

    y_train = train_df[
        TARGET
    ]

    X_validation = validation_df[
        MODEL_FEATURES
    ]

    y_validation = validation_df[
        TARGET
    ]

    results = []

    for alpha in alphas:

        model = Pipeline(
            [
                (
                    "scaler",
                    StandardScaler()
                ),
                (
                    "ridge",
                    Ridge(
                        alpha=alpha
                    )
                )
            ]
        )

        model.fit(
            X_train,
            y_train
        )

        pred = model.predict(
            X_validation
        )

        result_df = (
            validation_df[
                [
                    "signal_date",
                    "ticker",
                    TARGET
                ]
            ]
            .copy()
        )

        result_df[
            "prediction"
        ] = pred

        rmse = np.sqrt(
            mean_squared_error(
                y_validation,
                pred
            )
        )

        mae = mean_absolute_error(
            y_validation,
            pred
        )

        r2 = r2_score(
            y_validation,
            pred
        )

        ic_values = calculate_ic(
            result_df,
            "prediction",
            TARGET
        )

        results.append(
            {
                "alpha": alpha,
                "rmse": rmse,
                "mae": mae,
                "r2": r2,
                "mean_ic": ic_values.mean(),
                "median_ic": np.median(
                    ic_values
                ),
                "ic_signals": len(
                    ic_values
                )
            }
        )

    return pd.DataFrame(
        results
    )

In [21]:
# 06-18. 확장 alpha validation

ridge_validation_results = (
    evaluate_ridge_alphas(
        train_df,
        validation_df,
        RIDGE_ALPHAS
    )
)

print(
    ridge_validation_results.to_string(
        index=False
    )
)

      alpha     rmse      mae        r2   mean_ic  median_ic  ic_signals
      0.001 0.067651 0.047947 -0.066008 -0.004534  -0.059304          25
      0.010 0.067651 0.047947 -0.066007 -0.004534  -0.059304          25
      0.100 0.067651 0.047947 -0.066000 -0.004534  -0.059304          25
      1.000 0.067649 0.047943 -0.065927 -0.004424  -0.058343          25
     10.000 0.067626 0.047911 -0.065207 -0.005830  -0.051525          25
    100.000 0.067432 0.047626 -0.059092 -0.006048  -0.052485          25
   1000.000 0.066652 0.046436 -0.034732  0.003301  -0.034526          25
  10000.000 0.065758 0.044925 -0.007182 -0.010391   0.012629          25
 100000.000 0.065537 0.044394 -0.000412 -0.047646  -0.020312          25
1000000.000 0.065558 0.044376 -0.001069 -0.051911  -0.025402          25


In [22]:
# 06-19. mean baseline 비교
# dummyregressor: 실제 머신러닝 x, 항상 train 평균값만 예측하는 단순한 모델

from sklearn.dummy import DummyRegressor

dummy_model = DummyRegressor(
    strategy="mean"
)

dummy_model.fit(
    X_train,
    y_train
)

dummy_pred = dummy_model.predict(
    X_validation
)


dummy_rmse = np.sqrt(
    mean_squared_error(
        y_validation,
        dummy_pred
    )
)

dummy_mae = mean_absolute_error(
    y_validation,
    dummy_pred
)

dummy_r2 = r2_score(
    y_validation,
    dummy_pred
)


print(
    "dummy rmse:",
    dummy_rmse
)

print(
    "dummy mae:",
    dummy_mae
)

print(
    "dummy r2:",
    dummy_r2
)

dummy rmse: 0.06556366631432446
dummy mae: 0.04437601333543676
dummy r2: -0.0012281676811285447


Ridge RMSE < Dummy RMSE
→ feature에서 조금이라도 예측 정보를 뽑아냄

Ridge RMSE ≈ Dummy RMSE
→ 사실상 평균 예측과 비슷

Ridge RMSE > Dummy RMSE
→ 현재 Ridge가 평균 예측보다도 못함

In [23]:
# 06-20. best alpha 확정

best_alpha = (
    ridge_validation_results
    .loc[
        ridge_validation_results["rmse"].idxmin(),
        "alpha"
    ]
)

print(
    "best alpha:",
    best_alpha
)

best alpha: 100000.0


In [24]:
# 06-21. train validation 결합

train_validation_mask = (
    (
        supervised_dataset["signal_date"]
        < fold["test_start"]
    )
    &
    (
        supervised_dataset["next_execution_date"]
        <= fold["test_start"]
    )
)

test_mask = (
    (
        supervised_dataset["signal_date"]
        >= fold["test_start"]
    )
    &
    (
        supervised_dataset["signal_date"]
        < fold["test_end"]
    )
    &
    (
        supervised_dataset["next_execution_date"]
        <= fold["test_end"]
    )
)

train_validation_df = (
    supervised_dataset.loc[
        train_validation_mask
    ]
    .copy()
)

test_df = (
    supervised_dataset.loc[
        test_mask
    ]
    .copy()
)

print(
    "train + validation:",
    train_validation_df.shape
)

print(
    "test:",
    test_df.shape
)

print(
    "test signals:",
    test_df["signal_date"].nunique()
)

train + validation: (9089, 39)
test: (1250, 39)
test signals: 25


In [25]:
# 06-22. fold 1 ridge 최종 학습

X_train_validation = (
    train_validation_df[
        MODEL_FEATURES
    ]
)

y_train_validation = (
    train_validation_df[
        TARGET
    ]
)

X_test = (
    test_df[
        MODEL_FEATURES
    ]
)

y_test = (
    test_df[
        TARGET
    ]
)


ridge_model = Pipeline(
    [
        (
            "scaler",
            StandardScaler()
        ),
        (
            "ridge",
            Ridge(
                alpha=best_alpha
            )
        )
    ]
)


ridge_model.fit(
    X_train_validation,
    y_train_validation
)


test_pred = ridge_model.predict(
    X_test
)

In [26]:
# 06-23. fold 1 test 성능

test_result = (
    test_df[
        [
            "signal_date",
            "ticker",
            "name",
            TARGET
        ]
    ]
    .copy()
)

test_result["prediction"] = (
    test_pred
)


test_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        test_pred
    )
)

test_mae = mean_absolute_error(
    y_test,
    test_pred
)

test_r2 = r2_score(
    y_test,
    test_pred
)


test_ic_values = calculate_ic(
    test_result,
    "prediction",
    TARGET
)


print(
    "ridge test rmse:",
    test_rmse
)

print(
    "ridge test mae:",
    test_mae
)

print(
    "ridge test r2:",
    test_r2
)

print(
    "ridge mean ic:",
    test_ic_values.mean()
)

print(
    "ridge median ic:",
    np.median(
        test_ic_values
    )
)

print(
    "ic signals:",
    len(
        test_ic_values
    )
)

ridge test rmse: 0.06416200875265132
ridge test mae: 0.04719493397945572
ridge test r2: -0.013011136071444884
ridge mean ic: -0.05793997599039617
ridge median ic: -0.07812725090036014
ic signals: 25


In [27]:
# 06-24. fold 1 dummy test 비교

dummy_model = DummyRegressor(
    strategy="mean"
)

dummy_model.fit(
    X_train_validation,
    y_train_validation
)

dummy_test_pred = (
    dummy_model.predict(
        X_test
    )
)


dummy_test_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        dummy_test_pred
    )
)

dummy_test_mae = mean_absolute_error(
    y_test,
    dummy_test_pred
)

dummy_test_r2 = r2_score(
    y_test,
    dummy_test_pred
)


print(
    "dummy test rmse:",
    dummy_test_rmse
)

print(
    "dummy test mae:",
    dummy_test_mae
)

print(
    "dummy test r2:",
    dummy_test_r2
)

dummy test rmse: 0.06454794662386591
dummy test mae: 0.04762054947226941
dummy test r2: -0.025234418652478885


In [28]:
# 06-25. fold 1 ridge dummy 비교

fold1_comparison = pd.DataFrame(
    [
        {
            "model": "ridge",
            "rmse": test_rmse,
            "mae": test_mae,
            "r2": test_r2,
            "mean_ic": test_ic_values.mean(),
            "median_ic": np.median(
                test_ic_values
            )
        },
        {
            "model": "dummy_mean",
            "rmse": dummy_test_rmse,
            "mae": dummy_test_mae,
            "r2": dummy_test_r2,
            "mean_ic": np.nan,
            "median_ic": np.nan
        }
    ]
)

print(
    fold1_comparison.to_string(
        index=False
    )
)

     model     rmse      mae        r2  mean_ic  median_ic
     ridge 0.064162 0.047195 -0.013011 -0.05794  -0.078127
dummy_mean 0.064548 0.047621 -0.025234      NaN        NaN


Ridge
→ 선형 관계만 표현

Random Forest / XGBoost
→ 비선형 관계와 feature 간 interaction도 표현

In [37]:
# 06-26. ridge walk-forward 실행

ridge_fold_results = []
ridge_oos_prediction_frames = []


for fold in folds:

    train_mask = (
        (
            supervised_dataset["signal_date"]
            < fold["train_end"]
        )
        &
        (
            supervised_dataset["next_execution_date"]
            <= fold["train_end"]
        )
    )

    validation_mask = (
        (
            supervised_dataset["signal_date"]
            >= fold["validation_start"]
        )
        &
        (
            supervised_dataset["signal_date"]
            < fold["validation_end"]
        )
        &
        (
            supervised_dataset["next_execution_date"]
            <= fold["validation_end"]
        )
    )

    test_mask = (
        (
            supervised_dataset["signal_date"]
            >= fold["test_start"]
        )
        &
        (
            supervised_dataset["signal_date"]
            < fold["test_end"]
        )
        &
        (
            supervised_dataset["next_execution_date"]
            <= fold["test_end"]
        )
    )

    train_df = (
        supervised_dataset.loc[
            train_mask
        ]
        .copy()
    )

    validation_df = (
        supervised_dataset.loc[
            validation_mask
        ]
        .copy()
    )

    test_df = (
        supervised_dataset.loc[
            test_mask
        ]
        .copy()
    )

    validation_results = (
        evaluate_ridge_alphas(
            train_df,
            validation_df,
            RIDGE_ALPHAS
        )
    )

    best_row = (
        validation_results
        .loc[
            validation_results["rmse"]
            .idxmin()
        ]
    )

    best_alpha = (
        best_row["alpha"]
    )

    train_validation_mask = (
        (
            supervised_dataset["signal_date"]
            < fold["test_start"]
        )
        &
        (
            supervised_dataset["next_execution_date"]
            <= fold["test_start"]
        )
    )

    train_validation_df = (
        supervised_dataset.loc[
            train_validation_mask
        ]
        .copy()
    )

    X_train_validation = (
        train_validation_df[
            MODEL_FEATURES
        ]
    )

    y_train_validation = (
        train_validation_df[
            TARGET
        ]
    )

    X_test = (
        test_df[
            MODEL_FEATURES
        ]
    )

    y_test = (
        test_df[
            TARGET
        ]
    )

    ridge_model = Pipeline(
        [
            (
                "scaler",
                StandardScaler()
            ),
            (
                "ridge",
                Ridge(
                    alpha=best_alpha
                )
            )
        ]
    )

    ridge_model.fit(
        X_train_validation,
        y_train_validation
    )

    test_pred = (
        ridge_model.predict(
            X_test
        )
    )

    test_result = (
        test_df[
            [
                "signal_date",
                "execution_date",
                "ticker",
                "name",
                TARGET
            ]
        ]
        .copy()
    )

    test_result["prediction"] = (
        test_pred
    )

    test_result["fold"] = (
        fold["fold"]
    )

    ic_values = calculate_ic(
        test_result,
        "prediction",
        TARGET
    )

    test_rmse = np.sqrt(
        mean_squared_error(
            y_test,
            test_pred
        )
    )

    test_mae = mean_absolute_error(
        y_test,
        test_pred
    )

    test_r2 = r2_score(
        y_test,
        test_pred
    )

    dummy_model = DummyRegressor(
        strategy="mean"
    )

    dummy_model.fit(
        X_train_validation,
        y_train_validation
    )

    dummy_pred = (
        dummy_model.predict(
            X_test
        )
    )

    dummy_rmse = np.sqrt(
        mean_squared_error(
            y_test,
            dummy_pred
        )
    )

    dummy_mae = mean_absolute_error(
        y_test,
        dummy_pred
    )

    dummy_r2 = r2_score(
        y_test,
        dummy_pred
    )

    ridge_fold_results.append(
        {
            "fold": fold["fold"],
            "best_alpha": best_alpha,
            "test_start": fold["test_start"],
            "test_end": fold["test_end"],
            "test_rows": len(test_df),
            "test_signals": test_df[
                "signal_date"
            ].nunique(),
            "ridge_rmse": test_rmse,
            "ridge_mae": test_mae,
            "ridge_r2": test_r2,
            "mean_ic": ic_values.mean(),
            "median_ic": np.median(
                ic_values
            ),
            "dummy_rmse": dummy_rmse,
            "dummy_mae": dummy_mae,
            "dummy_r2": dummy_r2
        }
    )

    ridge_oos_prediction_frames.append(
        test_result
    )

fold마다

과거 train
↓
validation에서 alpha 선택
↓
train + validation 재학습
↓
미래 test 예측
↓
성능 저장
↓
6개월 앞으로 이동

In [30]:
# 06-27. ridge fold 결과 확인

ridge_fold_results = pd.DataFrame(
    ridge_fold_results
)

print(
    ridge_fold_results.to_string(
        index=False
    )
)

 fold  best_alpha test_start   test_end  test_rows  test_signals  ridge_rmse  ridge_mae  ridge_r2   mean_ic  median_ic  dummy_rmse  dummy_mae  dummy_r2
    1    100000.0 2021-11-04 2022-05-04       1250            25    0.064162   0.047195 -0.013011 -0.057940  -0.078127    0.064548   0.047621 -0.025234
    2     10000.0 2022-05-04 2022-11-04       1250            25    0.065222   0.047433 -0.018258 -0.072346  -0.071789    0.065276   0.047685 -0.019936
    3    100000.0 2022-11-04 2023-05-04       1248            25    0.074754   0.049259 -0.016305 -0.045032  -0.091285    0.074470   0.049080 -0.008575
    4   1000000.0 2023-05-04 2023-11-04       1250            25    0.076929   0.050800 -0.008933 -0.092552  -0.062953    0.076923   0.050796 -0.008781
    5   1000000.0 2023-11-04 2024-05-04       1197            24    0.076076   0.051550 -0.005253 -0.097931  -0.116639    0.076069   0.051544 -0.005060
    6   1000000.0 2024-05-04 2024-11-04       1249            25    0.081827   0.057125 

In [31]:
# 06-28. dummy 대비 ridge 성능 확인

ridge_fold_results[
    "rmse_improvement"
] = (
    ridge_fold_results[
        "dummy_rmse"
    ]
    - ridge_fold_results[
        "ridge_rmse"
    ]
)


ridge_fold_results[
    "ridge_better"
] = (
    ridge_fold_results[
        "ridge_rmse"
    ]
    <
    ridge_fold_results[
        "dummy_rmse"
    ]
)


print(
    ridge_fold_results[
        [
            "fold",
            "best_alpha",
            "ridge_rmse",
            "dummy_rmse",
            "rmse_improvement",
            "ridge_better",
            "mean_ic"
        ]
    ]
    .to_string(
        index=False
    )
)

 fold  best_alpha  ridge_rmse  dummy_rmse  rmse_improvement  ridge_better   mean_ic
    1    100000.0    0.064162    0.064548      3.859379e-04          True -0.057940
    2     10000.0    0.065222    0.065276      5.371359e-05          True -0.072346
    3    100000.0    0.074754    0.074470     -2.848387e-04         False -0.045032
    4   1000000.0    0.076929    0.076923     -5.816116e-06         False -0.092552
    5   1000000.0    0.076076    0.076069     -7.289964e-06         False -0.097931
    6   1000000.0    0.081827    0.081828      6.304131e-07          True -0.099560
    7   1000000.0    0.084041    0.084036     -4.985166e-06         False -0.067013
    8   1000000.0    0.079309    0.079276     -3.369801e-05         False -0.074316
    9   1000000.0    0.099569    0.099644      7.548953e-05          True -0.012025


Dummy RMSE - Ridge RMSE > 0
→ Ridge가 더 좋음

In [32]:
# 06-29. ridge walk-forward 요약

print(
    "mean ridge rmse:",
    ridge_fold_results[
        "ridge_rmse"
    ].mean()
)

print(
    "mean dummy rmse:",
    ridge_fold_results[
        "dummy_rmse"
    ].mean()
)

print(
    "mean ridge mae:",
    ridge_fold_results[
        "ridge_mae"
    ].mean()
)

print(
    "mean ridge r2:",
    ridge_fold_results[
        "ridge_r2"
    ].mean()
)

print(
    "mean ic:",
    ridge_fold_results[
        "mean_ic"
    ].mean()
)

print(
    "median ic:",
    ridge_fold_results[
        "median_ic"
    ].median()
)

print(
    "ridge better folds:",
    ridge_fold_results[
        "ridge_better"
    ].sum(),
    "/",
    len(
        ridge_fold_results
    )
)

mean ridge rmse: 0.07798776321686077
mean dummy rmse: 0.07800766804689824
mean ridge mae: 0.05447250699587663
mean ridge r2: -0.015348897611043978
mean ic: -0.06874613178604776
median ic: -0.08638655462184874
ridge better folds: 4 / 9


In [38]:
# 06-30. ridge oos prediction 결합

ridge_oos_predictions = (
    pd.concat(
        ridge_oos_prediction_frames,
        ignore_index=True
    )
    .sort_values(
        [
            "signal_date",
            "ticker"
        ]
    )
    .reset_index(
        drop=True
    )
)

print(
    "shape:",
    ridge_oos_predictions.shape
)

print(
    "signals:",
    ridge_oos_predictions[
        "signal_date"
    ].nunique()
)

print(
    "start:",
    ridge_oos_predictions[
        "signal_date"
    ].min()
)

print(
    "end:",
    ridge_oos_predictions[
        "signal_date"
    ].max()
)

shape: (11140, 7)
signals: 223
start: 2021-11-05 00:00:00
end: 2026-04-24 00:00:00


In [39]:
# 06-31. ridge alpha 범위 최종 설정

RIDGE_ALPHAS = [
    0.001,
    0.01,
    0.1,
    1.0,
    10.0,
    100.0,
    1000.0,
    10000.0,
    100000.0,
    1000000.0,
    10000000.0,
    100000000.0
]

alpha → 아주 커짐
      ↓
Ridge coefficient → 거의 0
      ↓
prediction → 거의 평균값
      ↓
DummyRegressor에 가까워짐

In [40]:
# 06-32. ridge 결과 저장 경로

RIDGE_RESULT_DIR = (
    PROJECT_ROOT
    / "data"
    / "predictions"
    / "ridge"
)

RIDGE_RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RIDGE_FOLD_PATH = (
    RIDGE_RESULT_DIR
    / "ridge_walk_forward_results.csv"
)

RIDGE_OOS_PATH = (
    RIDGE_RESULT_DIR
    / "ridge_oos_predictions.parquet"
)

In [42]:
# 06-32a. ridge fold 결과 dataframe 변환

ridge_fold_results = pd.DataFrame(
    ridge_fold_results
)

print(
    type(ridge_fold_results)
)

print(
    ridge_fold_results.shape
)

<class 'pandas.core.frame.DataFrame'>
(9, 14)


In [43]:
# 06-33. ridge fold 결과 저장

ridge_fold_results.to_csv(
    RIDGE_FOLD_PATH,
    index=False,
    encoding="utf-8-sig"
)

print(
    "saved:",
    RIDGE_FOLD_PATH
)

saved: C:\code\portfolio_optimization\data\predictions\ridge\ridge_walk_forward_results.csv


In [44]:
# 06-34. ridge oos prediction 저장

import pyarrow as pa

RIDGE_OOS_PATH = (
    RIDGE_RESULT_DIR
    / "ridge_oos_predictions.parquet"
)

ridge_oos_table = pa.Table.from_pandas(
    ridge_oos_predictions,
    preserve_index=False
)

pq.write_table(
    ridge_oos_table,
    RIDGE_OOS_PATH,
    compression="snappy"
)

print(
    "saved:",
    RIDGE_OOS_PATH
)

saved: C:\code\portfolio_optimization\data\predictions\ridge\ridge_oos_predictions.parquet


In [45]:
# 06-35. ridge 저장 결과 확인

ridge_oos_check = (
    pq.read_table(
        RIDGE_OOS_PATH
    )
    .to_pandas()
)

print(
    "shape:",
    ridge_oos_check.shape
)

print(
    "signals:",
    ridge_oos_check[
        "signal_date"
    ].nunique()
)

print(
    "start:",
    ridge_oos_check[
        "signal_date"
    ].min()
)

print(
    "end:",
    ridge_oos_check[
        "signal_date"
    ].max()
)

print(
    "duplicates:",
    ridge_oos_check[
        [
            "signal_date",
            "ticker"
        ]
    ]
    .duplicated()
    .sum()
)

shape: (11140, 7)
signals: 223
start: 2021-11-05 00:00:00
end: 2026-04-24 00:00:00
duplicates: 0


In [46]:
# 06-36. ridge prediction 분산 확인

ridge_prediction_stats = (
    ridge_oos_predictions
    .groupby(
        "signal_date"
    )
    .agg(
        prediction_std=(
            "prediction",
            "std"
        ),
        target_std=(
            TARGET,
            "std"
        )
    )
)

print(
    ridge_prediction_stats.describe()
)

       prediction_std  target_std
count      223.000000  223.000000
mean         0.000287    0.068816
std          0.000361    0.020857
min          0.000038    0.034853
25%          0.000070    0.051816
50%          0.000094    0.064346
75%          0.000432    0.082868
max          0.001906    0.137248


여기서 prediction_std가 target_std보다 굉장히 작다면:
실제 종목 수익률
→ 종목마다 크게 다름

Ridge prediction
→ 종목별 차이가 거의 없음

Ridge의 성격이 거의 확정적인 듯

평균 target std      = 0.068816  ≈ 6.88%
평균 prediction std  = 0.000287  ≈ 0.029%

즉 실제로는 같은 주에 50개 종목 수익률이 꽤 넓게 퍼지는데,
Ridge는 종목별 예측을 거의 똑같이 내고 있음. 실제 변동폭이 예측 변동폭보다 약 240배 큼

쉽게 말하면 Ridge가:
삼성전자   +0.12%
SK하이닉스 +0.13%
NAVER      +0.11%
카카오     +0.12%
...
처럼 거의 평균값 근처만 내놓는 상황
지금까지 나온 큰 alpha + Dummy와 거의 같은 RMSE + 음수 IC가 서로 일관된 결과임.

In [47]:
# 06-37. random forest 패키지

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import ParameterGrid

In [48]:
# 06-38. fold 1 데이터 설정

fold = folds[0]

train_mask = (
    (supervised_dataset["signal_date"] < fold["train_end"])
    &
    (
        supervised_dataset["next_execution_date"]
        <= fold["train_end"]
    )
)

validation_mask = (
    (
        supervised_dataset["signal_date"]
        >= fold["validation_start"]
    )
    &
    (
        supervised_dataset["signal_date"]
        < fold["validation_end"]
    )
    &
    (
        supervised_dataset["next_execution_date"]
        <= fold["validation_end"]
    )
)

train_df = (
    supervised_dataset.loc[
        train_mask
    ]
    .copy()
)

validation_df = (
    supervised_dataset.loc[
        validation_mask
    ]
    .copy()
)

print(
    "train:",
    train_df.shape
)

print(
    "validation:",
    validation_df.shape
)

train: (7791, 39)
validation: (1248, 39)


In [49]:
# 06-39. random forest 후보 설정

RF_PARAM_GRID = {
    "max_depth": [
        3,
        5,
        8
    ],
    "min_samples_leaf": [
        20,
        50,
        100
    ]
}

rf_candidates = list(
    ParameterGrid(
        RF_PARAM_GRID
    )
)

print(
    "candidates:",
    len(rf_candidates)
)

candidates: 9


max_depth
= tree 최대 깊이
= 클수록 복잡한 관계 표현

min_samples_leaf
= tree 마지막 leaf에 최소 몇 개 sample을 남길지
= 클수록 과적합 억제

In [50]:
# 06-40. random forest validation

X_train = train_df[
    MODEL_FEATURES
]

y_train = train_df[
    TARGET
]

X_validation = validation_df[
    MODEL_FEATURES
]

y_validation = validation_df[
    TARGET
]


rf_validation_results = []


for params in rf_candidates:

    model = RandomForestRegressor(
        n_estimators=300,
        max_depth=params["max_depth"],
        min_samples_leaf=params[
            "min_samples_leaf"
        ],
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train,
        y_train
    )

    pred = model.predict(
        X_validation
    )

    result_df = (
        validation_df[
            [
                "signal_date",
                "ticker",
                TARGET
            ]
        ]
        .copy()
    )

    result_df[
        "prediction"
    ] = pred


    rmse = np.sqrt(
        mean_squared_error(
            y_validation,
            pred
        )
    )

    mae = mean_absolute_error(
        y_validation,
        pred
    )

    r2 = r2_score(
        y_validation,
        pred
    )

    ic_values = calculate_ic(
        result_df,
        "prediction",
        TARGET
    )


    rf_validation_results.append(
        {
            "max_depth": params[
                "max_depth"
            ],
            "min_samples_leaf": params[
                "min_samples_leaf"
            ],
            "rmse": rmse,
            "mae": mae,
            "r2": r2,
            "mean_ic": ic_values.mean(),
            "median_ic": np.median(
                ic_values
            )
        }
    )


rf_validation_results = pd.DataFrame(
    rf_validation_results
)


print(
    rf_validation_results
    .sort_values(
        "rmse"
    )
    .to_string(
        index=False
    )
)

 max_depth  min_samples_leaf     rmse      mae        r2   mean_ic  median_ic
         8               100 0.065319 0.043657  0.006245  0.077991   0.060840
         5               100 0.065362 0.043735  0.004920 -0.005032  -0.004274
         5                20 0.065394 0.043776  0.003956  0.052619   0.101095
         3               100 0.065404 0.043933  0.003631  0.044001   0.065932
         3                20 0.065521 0.043966  0.000064  0.031967   0.039919
         8                20 0.065610 0.043876 -0.002654  0.073415   0.111068
         8                50 0.065703 0.044009 -0.005503  0.067478   0.079288
         5                50 0.065881 0.044259 -0.010958  0.068295   0.085915
         3                50 0.065935 0.044431 -0.012589  0.002549   0.023234


In [51]:
# 06-41. best random forest 선택

best_rf_row = (
    rf_validation_results
    .loc[
        rf_validation_results[
            "rmse"
        ]
        .idxmin()
    ]
)

best_rf_params = {
    "max_depth": int(
        best_rf_row[
            "max_depth"
        ]
    ),
    "min_samples_leaf": int(
        best_rf_row[
            "min_samples_leaf"
        ]
    )
}


print(
    "best params:",
    best_rf_params
)

print(
    best_rf_row
)

best params: {'max_depth': 8, 'min_samples_leaf': 100}
max_depth             8.000000
min_samples_leaf    100.000000
rmse                  0.065319
mae                   0.043657
r2                    0.006245
mean_ic               0.077991
median_ic             0.060840
Name: 8, dtype: float64


In [52]:
# 06-42. random forest 후보 확장

RF_PARAM_GRID = {
    "max_depth": [
        5,
        8,
        12,
        None
    ],
    "min_samples_leaf": [
        50,
        100,
        150,
        200
    ]
}

rf_candidates = list(
    ParameterGrid(
        RF_PARAM_GRID
    )
)

print(
    "candidates:",
    len(rf_candidates)
)

candidates: 16


In [53]:
# 06-43. random forest validation 함수

def evaluate_rf_params(
    train_df,
    validation_df,
    candidates
):

    X_train = train_df[
        MODEL_FEATURES
    ]

    y_train = train_df[
        TARGET
    ]

    X_validation = validation_df[
        MODEL_FEATURES
    ]

    y_validation = validation_df[
        TARGET
    ]

    results = []

    for params in candidates:

        model = RandomForestRegressor(
            n_estimators=300,
            max_depth=params["max_depth"],
            min_samples_leaf=params[
                "min_samples_leaf"
            ],
            max_features="sqrt",
            random_state=42,
            n_jobs=-1
        )

        model.fit(
            X_train,
            y_train
        )

        pred = model.predict(
            X_validation
        )

        result_df = (
            validation_df[
                [
                    "signal_date",
                    "ticker",
                    TARGET
                ]
            ]
            .copy()
        )

        result_df["prediction"] = pred

        rmse = np.sqrt(
            mean_squared_error(
                y_validation,
                pred
            )
        )

        mae = mean_absolute_error(
            y_validation,
            pred
        )

        r2 = r2_score(
            y_validation,
            pred
        )

        ic_values = calculate_ic(
            result_df,
            "prediction",
            TARGET
        )

        results.append(
            {
                "max_depth": params[
                    "max_depth"
                ],
                "min_samples_leaf": params[
                    "min_samples_leaf"
                ],
                "rmse": rmse,
                "mae": mae,
                "r2": r2,
                "mean_ic": ic_values.mean(),
                "median_ic": np.median(
                    ic_values
                )
            }
        )

    return pd.DataFrame(
        results
    )

In [54]:
# 06-44. 확장 random forest validation

rf_validation_results = (
    evaluate_rf_params(
        train_df,
        validation_df,
        rf_candidates
    )
)

print(
    rf_validation_results
    .sort_values(
        "rmse"
    )
    .to_string(
        index=False
    )
)

 max_depth  min_samples_leaf     rmse      mae        r2   mean_ic  median_ic
       8.0               100 0.065319 0.043657  0.006245  0.077991   0.060840
       5.0               150 0.065355 0.043842  0.005126  0.045590   0.035102
       5.0               100 0.065362 0.043735  0.004920 -0.005032  -0.004274
       8.0               200 0.065378 0.043831  0.004425  0.012592  -0.032509
       5.0               200 0.065388 0.043917  0.004130  0.017180   0.032366
       NaN               150 0.065418 0.043773  0.003221  0.069105   0.049892
      12.0               150 0.065418 0.043797  0.003202  0.067142   0.076591
       NaN               200 0.065432 0.043874  0.002791  0.033944  -0.011765
      12.0               200 0.065433 0.043874  0.002770  0.034791  -0.010132
       8.0               150 0.065440 0.043856  0.002536  0.051046   0.055942
       NaN               100 0.065444 0.043732  0.002438  0.027627   0.020888
      12.0               100 0.065457 0.043684  0.002029  0.0362

In [55]:
# 06-45. best random forest 확정

best_rf_row = (
    rf_validation_results
    .loc[
        rf_validation_results[
            "rmse"
        ]
        .idxmin()
    ]
)

best_rf_params = {
    "max_depth": (
        None
        if pd.isna(
            best_rf_row["max_depth"]
        )
        else int(
            best_rf_row["max_depth"]
        )
    ),
    "min_samples_leaf": int(
        best_rf_row[
            "min_samples_leaf"
        ]
    )
}

print(
    "best params:",
    best_rf_params
)

print(
    best_rf_row
)

best params: {'max_depth': 8, 'min_samples_leaf': 100}
max_depth             8.000000
min_samples_leaf    100.000000
rmse                  0.065319
mae                   0.043657
r2                    0.006245
mean_ic               0.077991
median_ic             0.060840
Name: 5, dtype: float64


In [56]:
# 06-46. fold 1 최종 데이터 설정

fold = folds[0]

train_validation_mask = (
    (
        supervised_dataset["signal_date"]
        < fold["test_start"]
    )
    &
    (
        supervised_dataset["next_execution_date"]
        <= fold["test_start"]
    )
)

test_mask = (
    (
        supervised_dataset["signal_date"]
        >= fold["test_start"]
    )
    &
    (
        supervised_dataset["signal_date"]
        < fold["test_end"]
    )
    &
    (
        supervised_dataset["next_execution_date"]
        <= fold["test_end"]
    )
)

train_validation_df = (
    supervised_dataset.loc[
        train_validation_mask
    ]
    .copy()
)

test_df = (
    supervised_dataset.loc[
        test_mask
    ]
    .copy()
)

print(
    "train + validation:",
    train_validation_df.shape
)

print(
    "test:",
    test_df.shape
)

print(
    "test signals:",
    test_df["signal_date"].nunique()
)

train + validation: (9089, 39)
test: (1250, 39)
test signals: 25


In [57]:
# 06-47. fold 1 random forest 최종 학습

X_train_validation = (
    train_validation_df[
        MODEL_FEATURES
    ]
)

y_train_validation = (
    train_validation_df[
        TARGET
    ]
)

X_test = (
    test_df[
        MODEL_FEATURES
    ]
)

y_test = (
    test_df[
        TARGET
    ]
)

rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=best_rf_params[
        "max_depth"
    ],
    min_samples_leaf=best_rf_params[
        "min_samples_leaf"
    ],
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(
    X_train_validation,
    y_train_validation
)

rf_test_pred = rf_model.predict(
    X_test
)

In [58]:
# 06-48. fold 1 random forest test 성능

rf_test_result = (
    test_df[
        [
            "signal_date",
            "ticker",
            "name",
            TARGET
        ]
    ]
    .copy()
)

rf_test_result[
    "prediction"
] = rf_test_pred


rf_test_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        rf_test_pred
    )
)

rf_test_mae = mean_absolute_error(
    y_test,
    rf_test_pred
)

rf_test_r2 = r2_score(
    y_test,
    rf_test_pred
)

rf_test_ic_values = calculate_ic(
    rf_test_result,
    "prediction",
    TARGET
)


print(
    "rf test rmse:",
    rf_test_rmse
)

print(
    "rf test mae:",
    rf_test_mae
)

print(
    "rf test r2:",
    rf_test_r2
)

print(
    "rf mean ic:",
    rf_test_ic_values.mean()
)

print(
    "rf median ic:",
    np.median(
        rf_test_ic_values
    )
)

print(
    "ic signals:",
    len(
        rf_test_ic_values
    )
)

rf test rmse: 0.06402979754678921
rf test mae: 0.04707430952573367
rf test r2: -0.008840648333260148
rf mean ic: -0.026254895194565395
rf median ic: -0.022136854741896757
ic signals: 25


In [59]:
# 06-49. fold 1 model 비교

fold1_model_comparison = pd.DataFrame(
    [
        {
            "model": "dummy_mean",
            "rmse": dummy_test_rmse,
            "mae": dummy_test_mae,
            "r2": dummy_test_r2,
            "mean_ic": np.nan,
            "median_ic": np.nan
        },
        {
            "model": "ridge",
            "rmse": test_rmse,
            "mae": test_mae,
            "r2": test_r2,
            "mean_ic": test_ic_values.mean(),
            "median_ic": np.median(
                test_ic_values
            )
        },
        {
            "model": "random_forest",
            "rmse": rf_test_rmse,
            "mae": rf_test_mae,
            "r2": rf_test_r2,
            "mean_ic": rf_test_ic_values.mean(),
            "median_ic": np.median(
                rf_test_ic_values
            )
        }
    ]
)

print(
    fold1_model_comparison.to_string(
        index=False
    )
)

        model     rmse      mae        r2   mean_ic  median_ic
   dummy_mean 0.064548 0.047621 -0.025234       NaN        NaN
        ridge 0.099569 0.068568 -0.030254 -0.057940  -0.078127
random_forest 0.064030 0.047074 -0.008841 -0.026255  -0.022137


In [60]:
# 06-50. random forest prediction 분산 확인

rf_prediction_stats = (
    rf_test_result
    .groupby(
        "signal_date"
    )
    .agg(
        prediction_std=(
            "prediction",
            "std"
        ),
        target_std=(
            TARGET,
            "std"
        )
    )
)

print(
    rf_prediction_stats.describe()
)

       prediction_std  target_std
count       25.000000   25.000000
mean         0.001311    0.058223
std          0.000319    0.010933
min          0.000777    0.040786
25%          0.001091    0.050320
50%          0.001275    0.056460
75%          0.001521    0.063934
max          0.001948    0.086535


In [61]:
# 06-51. fold 1 model 비교 수정

ridge_fold1 = (
    ridge_fold_results[
        ridge_fold_results["fold"] == 1
    ]
    .iloc[0]
)

fold1_model_comparison = pd.DataFrame(
    [
        {
            "model": "dummy_mean",
            "rmse": dummy_test_rmse,
            "mae": dummy_test_mae,
            "r2": dummy_test_r2,
            "mean_ic": np.nan,
            "median_ic": np.nan
        },
        {
            "model": "ridge",
            "rmse": ridge_fold1[
                "ridge_rmse"
            ],
            "mae": ridge_fold1[
                "ridge_mae"
            ],
            "r2": ridge_fold1[
                "ridge_r2"
            ],
            "mean_ic": ridge_fold1[
                "mean_ic"
            ],
            "median_ic": ridge_fold1[
                "median_ic"
            ]
        },
        {
            "model": "random_forest",
            "rmse": rf_test_rmse,
            "mae": rf_test_mae,
            "r2": rf_test_r2,
            "mean_ic": rf_test_ic_values.mean(),
            "median_ic": np.median(
                rf_test_ic_values
            )
        }
    ]
)

print(
    fold1_model_comparison.to_string(
        index=False
    )
)

        model     rmse      mae        r2   mean_ic  median_ic
   dummy_mean 0.064548 0.047621 -0.025234       NaN        NaN
        ridge 0.064162 0.047195 -0.013011 -0.057940  -0.078127
random_forest 0.064030 0.047074 -0.008841 -0.026255  -0.022137


In [62]:
# 06-52. random forest walk-forward 실행

rf_fold_result_rows = []
rf_oos_prediction_frames = []


for fold in folds:

    train_mask = (
        (
            supervised_dataset["signal_date"]
            < fold["train_end"]
        )
        &
        (
            supervised_dataset["next_execution_date"]
            <= fold["train_end"]
        )
    )

    validation_mask = (
        (
            supervised_dataset["signal_date"]
            >= fold["validation_start"]
        )
        &
        (
            supervised_dataset["signal_date"]
            < fold["validation_end"]
        )
        &
        (
            supervised_dataset["next_execution_date"]
            <= fold["validation_end"]
        )
    )

    test_mask = (
        (
            supervised_dataset["signal_date"]
            >= fold["test_start"]
        )
        &
        (
            supervised_dataset["signal_date"]
            < fold["test_end"]
        )
        &
        (
            supervised_dataset["next_execution_date"]
            <= fold["test_end"]
        )
    )

    train_df = (
        supervised_dataset.loc[
            train_mask
        ]
        .copy()
    )

    validation_df = (
        supervised_dataset.loc[
            validation_mask
        ]
        .copy()
    )

    test_df = (
        supervised_dataset.loc[
            test_mask
        ]
        .copy()
    )

    validation_results = (
        evaluate_rf_params(
            train_df,
            validation_df,
            rf_candidates
        )
    )

    best_row = (
        validation_results
        .loc[
            validation_results[
                "rmse"
            ]
            .idxmin()
        ]
    )

    best_max_depth = (
        None
        if pd.isna(
            best_row["max_depth"]
        )
        else int(
            best_row["max_depth"]
        )
    )

    best_min_samples_leaf = int(
        best_row[
            "min_samples_leaf"
        ]
    )

    train_validation_mask = (
        (
            supervised_dataset["signal_date"]
            < fold["test_start"]
        )
        &
        (
            supervised_dataset["next_execution_date"]
            <= fold["test_start"]
        )
    )

    train_validation_df = (
        supervised_dataset.loc[
            train_validation_mask
        ]
        .copy()
    )

    X_train_validation = (
        train_validation_df[
            MODEL_FEATURES
        ]
    )

    y_train_validation = (
        train_validation_df[
            TARGET
        ]
    )

    X_test = (
        test_df[
            MODEL_FEATURES
        ]
    )

    y_test = (
        test_df[
            TARGET
        ]
    )

    rf_model = RandomForestRegressor(
        n_estimators=300,
        max_depth=best_max_depth,
        min_samples_leaf=best_min_samples_leaf,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    )

    rf_model.fit(
        X_train_validation,
        y_train_validation
    )

    test_pred = (
        rf_model.predict(
            X_test
        )
    )

    test_result = (
        test_df[
            [
                "signal_date",
                "execution_date",
                "ticker",
                "name",
                TARGET
            ]
        ]
        .copy()
    )

    test_result["prediction"] = (
        test_pred
    )

    test_result["fold"] = (
        fold["fold"]
    )

    ic_values = calculate_ic(
        test_result,
        "prediction",
        TARGET
    )

    test_rmse = np.sqrt(
        mean_squared_error(
            y_test,
            test_pred
        )
    )

    test_mae = mean_absolute_error(
        y_test,
        test_pred
    )

    test_r2 = r2_score(
        y_test,
        test_pred
    )

    rf_fold_result_rows.append(
        {
            "fold": fold["fold"],
            "max_depth": best_max_depth,
            "min_samples_leaf":
                best_min_samples_leaf,
            "test_start": fold["test_start"],
            "test_end": fold["test_end"],
            "test_rows": len(test_df),
            "test_signals": test_df[
                "signal_date"
            ].nunique(),
            "rf_rmse": test_rmse,
            "rf_mae": test_mae,
            "rf_r2": test_r2,
            "mean_ic": ic_values.mean(),
            "median_ic": np.median(
                ic_values
            )
        }
    )

    rf_oos_prediction_frames.append(
        test_result
    )

전진 최적화(Walk-Forward Optimization)는 과거 데이터로 매매 전략의 매개변수를 최적화하고 검증하여 과최적화를 막는 금융·알고리즘 트레이딩 기법

In [63]:
# 06-53. random forest fold 결과 생성

rf_fold_results = pd.DataFrame(
    rf_fold_result_rows
)

print(
    rf_fold_results.to_string(
        index=False
    )
)

 fold  max_depth  min_samples_leaf test_start   test_end  test_rows  test_signals  rf_rmse   rf_mae     rf_r2   mean_ic  median_ic
    1        8.0               100 2021-11-04 2022-05-04       1250            25 0.064030 0.047074 -0.008841 -0.026255  -0.022137
    2        NaN               100 2022-05-04 2022-11-04       1250            25 0.070345 0.052272 -0.184486  0.001285   0.023866
    3        5.0               200 2022-11-04 2023-05-04       1248            25 0.074494 0.048914 -0.009242 -0.009196  -0.004274
    4        NaN                50 2023-05-04 2023-11-04       1250            25 0.078037 0.051983 -0.038215  0.016544   0.036831
    5        5.0                50 2023-11-04 2024-05-04       1197            24 0.076172 0.051632 -0.007795 -0.091607  -0.103097
    6        NaN                50 2024-05-04 2024-11-04       1249            25 0.081098 0.056450  0.017439  0.077777   0.050084
    7        NaN                50 2024-11-04 2025-05-04       1198            24 0

In [64]:
# 06-54. ridge random forest fold 비교

model_fold_comparison = (
    ridge_fold_results[
        [
            "fold",
            "ridge_rmse",
            "ridge_mae",
            "ridge_r2",
            "mean_ic"
        ]
    ]
    .rename(
        columns={
            "mean_ic":
            "ridge_mean_ic"
        }
    )
    .merge(
        rf_fold_results[
            [
                "fold",
                "rf_rmse",
                "rf_mae",
                "rf_r2",
                "mean_ic"
            ]
        ]
        .rename(
            columns={
                "mean_ic":
                "rf_mean_ic"
            }
        ),
        on="fold",
        how="inner"
    )
)

model_fold_comparison[
    "rf_rmse_improvement"
] = (
    model_fold_comparison[
        "ridge_rmse"
    ]
    - model_fold_comparison[
        "rf_rmse"
    ]
)

print(
    model_fold_comparison.to_string(
        index=False
    )
)

 fold  ridge_rmse  ridge_mae  ridge_r2  ridge_mean_ic  rf_rmse   rf_mae     rf_r2  rf_mean_ic  rf_rmse_improvement
    1    0.064162   0.047195 -0.013011      -0.057940 0.064030 0.047074 -0.008841   -0.026255             0.000132
    2    0.065222   0.047433 -0.018258      -0.072346 0.070345 0.052272 -0.184486    0.001285            -0.005123
    3    0.074754   0.049259 -0.016305      -0.045032 0.074494 0.048914 -0.009242   -0.009196             0.000260
    4    0.076929   0.050800 -0.008933      -0.092552 0.078037 0.051983 -0.038215    0.016544            -0.001108
    5    0.076076   0.051550 -0.005253      -0.097931 0.076172 0.051632 -0.007795   -0.091607            -0.000096
    6    0.081827   0.057125 -0.000319      -0.099560 0.081098 0.056450  0.017439    0.077777             0.000730
    7    0.084041   0.062017 -0.000132      -0.067013 0.084084 0.062164 -0.001147   -0.018259            -0.000043
    8    0.079309   0.056308 -0.045675      -0.074316 0.080157 0.057289 -0.06814

rf_rmse_improvement > 0
→ RF가 Ridge보다 RMSE 낮음

rf_rmse_improvement < 0
→ Ridge가 더 낮음

In [65]:
# 06-55. random forest walk-forward 요약

print(
    "mean rf rmse:",
    rf_fold_results[
        "rf_rmse"
    ].mean()
)

print(
    "mean ridge rmse:",
    ridge_fold_results[
        "ridge_rmse"
    ].mean()
)

print(
    "mean rf mae:",
    rf_fold_results[
        "rf_mae"
    ].mean()
)

print(
    "mean rf r2:",
    rf_fold_results[
        "rf_r2"
    ].mean()
)

print(
    "mean rf ic:",
    rf_fold_results[
        "mean_ic"
    ].mean()
)

print(
    "median rf ic:",
    rf_fold_results[
        "median_ic"
    ].median()
)

print(
    "rf better than ridge:",
    (
        model_fold_comparison[
            "rf_rmse_improvement"
        ]
        > 0
    ).sum(),
    "/",
    len(
        model_fold_comparison
    )
)

mean rf rmse: 0.07851099347628682
mean ridge rmse: 0.07798776321686077
mean rf mae: 0.055075577214919826
mean rf r2: -0.0335777054433235
mean rf ic: -0.00799505772887482
median rf ic: 0.023865546218487393
rf better than ridge: 4 / 9


In [66]:
# 06-56. random forest oos prediction 결합

rf_oos_predictions = (
    pd.concat(
        rf_oos_prediction_frames,
        ignore_index=True
    )
    .sort_values(
        [
            "signal_date",
            "ticker"
        ]
    )
    .reset_index(
        drop=True
    )
)

print(
    "shape:",
    rf_oos_predictions.shape
)

print(
    "signals:",
    rf_oos_predictions[
        "signal_date"
    ].nunique()
)

print(
    "start:",
    rf_oos_predictions[
        "signal_date"
    ].min()
)

print(
    "end:",
    rf_oos_predictions[
        "signal_date"
    ].max()
)

print(
    "duplicates:",
    rf_oos_predictions[
        [
            "signal_date",
            "ticker"
        ]
    ]
    .duplicated()
    .sum()
)

shape: (11140, 7)
signals: 223
start: 2021-11-05 00:00:00
end: 2026-04-24 00:00:00
duplicates: 0


In [67]:
# 06-57. random forest 결과 저장

RF_RESULT_DIR = (
    PROJECT_ROOT
    / "data"
    / "predictions"
    / "random_forest"
)

RF_RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RF_FOLD_PATH = (
    RF_RESULT_DIR
    / "rf_walk_forward_results.csv"
)

RF_OOS_PATH = (
    RF_RESULT_DIR
    / "rf_oos_predictions.parquet"
)

rf_fold_results.to_csv(
    RF_FOLD_PATH,
    index=False,
    encoding="utf-8-sig"
)

rf_oos_table = pa.Table.from_pandas(
    rf_oos_predictions,
    preserve_index=False
)

pq.write_table(
    rf_oos_table,
    RF_OOS_PATH,
    compression="snappy"
)

print(
    "saved:",
    RF_FOLD_PATH
)

print(
    "saved:",
    RF_OOS_PATH
)

saved: C:\code\portfolio_optimization\data\predictions\random_forest\rf_walk_forward_results.csv
saved: C:\code\portfolio_optimization\data\predictions\random_forest\rf_oos_predictions.parquet


Random Forest
tree 여러 개를 독립적으로 만듦 -> 결과 평균

XGBoost
첫 번째 tree가 틀린 부분 -> 다음 tree가 그 오류를 보완 -> 또 틀린 부분을 다음 tree가 보완 -> 순차적으로 개선

In [70]:
%pip install xgboost

  Using cached xgboost-3.2.0-py3-none-win_amd64.whl.metadata (2.1 kB)
Using cached xgboost-3.2.0-py3-none-win_amd64.whl (101.7 MB)
Note: you may need to restart the kernel to use updated packages.


In [71]:
# 06-58. xgboost 패키지 확인

import xgboost as xgb

from xgboost import XGBRegressor

print(
    "xgboost version:",
    xgb.__version__
)

xgboost version: 3.2.0


In [72]:
# 06-59. xgboost fold 1 데이터 설정

fold = folds[0]

train_mask = (
    (
        supervised_dataset["signal_date"]
        < fold["train_end"]
    )
    &
    (
        supervised_dataset["next_execution_date"]
        <= fold["train_end"]
    )
)

validation_mask = (
    (
        supervised_dataset["signal_date"]
        >= fold["validation_start"]
    )
    &
    (
        supervised_dataset["signal_date"]
        < fold["validation_end"]
    )
    &
    (
        supervised_dataset["next_execution_date"]
        <= fold["validation_end"]
    )
)

train_df = (
    supervised_dataset.loc[
        train_mask
    ]
    .copy()
)

validation_df = (
    supervised_dataset.loc[
        validation_mask
    ]
    .copy()
)

print(
    "train:",
    train_df.shape
)

print(
    "validation:",
    validation_df.shape
)

train: (7791, 39)
validation: (1248, 39)


In [73]:
# 06-60. xgboost 후보 설정

XGB_PARAM_GRID = {
    "max_depth": [
        2,
        3,
        5
    ],
    "learning_rate": [
        0.01,
        0.05
    ],
    "min_child_weight": [
        20,
        50
    ]
}

xgb_candidates = list(
    ParameterGrid(
        XGB_PARAM_GRID
    )
)

print(
    "candidates:",
    len(xgb_candidates)
)

candidates: 12


max_depth
= tree 최대 깊이

learning_rate
= 한 tree가 기존 예측을 얼마나 세게 수정할지
= 작을수록 조금씩 보정

min_child_weight
= leaf를 너무 작은 표본으로 쪼개지 못하게 막는 규제
= 클수록 보수적

In [74]:
# 06-61. xgboost validation 함수

def evaluate_xgb_params(
    train_df,
    validation_df,
    candidates
):

    X_train = train_df[
        MODEL_FEATURES
    ]

    y_train = train_df[
        TARGET
    ]

    X_validation = validation_df[
        MODEL_FEATURES
    ]

    y_validation = validation_df[
        TARGET
    ]

    results = []

    for params in candidates:

        model = XGBRegressor(
            n_estimators=300,
            max_depth=params[
                "max_depth"
            ],
            learning_rate=params[
                "learning_rate"
            ],
            min_child_weight=params[
                "min_child_weight"
            ],
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1
        )

        model.fit(
            X_train,
            y_train
        )

        pred = model.predict(
            X_validation
        )

        result_df = (
            validation_df[
                [
                    "signal_date",
                    "ticker",
                    TARGET
                ]
            ]
            .copy()
        )

        result_df[
            "prediction"
        ] = pred

        rmse = np.sqrt(
            mean_squared_error(
                y_validation,
                pred
            )
        )

        mae = mean_absolute_error(
            y_validation,
            pred
        )

        r2 = r2_score(
            y_validation,
            pred
        )

        ic_values = calculate_ic(
            result_df,
            "prediction",
            TARGET
        )

        results.append(
            {
                "max_depth": params[
                    "max_depth"
                ],
                "learning_rate": params[
                    "learning_rate"
                ],
                "min_child_weight": params[
                    "min_child_weight"
                ],
                "rmse": rmse,
                "mae": mae,
                "r2": r2,
                "mean_ic": ic_values.mean(),
                "median_ic": np.median(
                    ic_values
                )
            }
        )

    return pd.DataFrame(
        results
    )

In [75]:
# 06-62. xgboost fold 1 validation

xgb_validation_results = (
    evaluate_xgb_params(
        train_df,
        validation_df,
        xgb_candidates
    )
)

print(
    xgb_validation_results
    .sort_values(
        "rmse"
    )
    .to_string(
        index=False
    )
)

C:\Users\SD1-06\miniforge3\envs\finrl\lib\site-packages\numpy\lib\_function_base_impl.py:3045: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\SD1-06\miniforge3\envs\finrl\lib\site-packages\numpy\lib\_function_base_impl.py:3046: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


 max_depth  learning_rate  min_child_weight     rmse      mae        r2  mean_ic  median_ic
         3           0.01                20 0.065570 0.043954 -0.001428 0.037986   0.046360
         2           0.01                20 0.065727 0.044118 -0.006219 0.058418   0.027718
         2           0.05                20 0.065755 0.044710 -0.007084 0.013022  -0.024754
         3           0.05                20 0.065784 0.044678 -0.007983 0.028265   0.051911
         5           0.01                20 0.065790 0.044284 -0.008139 0.003817   0.007347
         3           0.05                50 0.066065 0.044871 -0.016590 0.004766   0.005138
         2           0.01                50 0.066120 0.044515 -0.018306 0.036998   0.061487
         3           0.01                50 0.066263 0.044620 -0.022694 0.003773  -0.006125
         5           0.01                50 0.066427 0.044836 -0.027774 0.009415  -0.044412
         2           0.05                50 0.066474 0.045200 -0.029221 0.006453

In [76]:
# 06-63. cross-sectional ic 함수 수정
# nunique = number of unique values = 고유값 개수

def calculate_ic(
    data,
    prediction_column,
    target_column
):

    ic_values = []

    for _, group in data.groupby(
        "signal_date"
    ):

        if len(group) < 2:
            continue

        if (
            group[prediction_column].nunique() < 2
            or
            group[target_column].nunique() < 2
        ):
            continue

        ic = (
            group[
                prediction_column
            ]
            .rank()
            .corr(
                group[
                    target_column
                ]
                .rank()
            )
        )

        if pd.notna(ic):
            ic_values.append(ic)

    return np.array(
        ic_values
    )